In [2]:
# IMPORTING LIBRARIES

import os
import zipfile
import json

from google.colab import drive

In [3]:
# ENVIRONMENT SETUP

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

BASE = '/content/drive/My Drive/project'

PATHS = {
    "ROBUST":   os.path.join(BASE, 'rob.zip'),
}

Mounted at /content/drive


In [4]:
#  ROBUST-MIPS — JSON STRUCTURE INSPECTION
#  Goal: identify the real annotation schema before parsing

zip_path = PATHS["ROBUST"]

with zipfile.ZipFile(zip_path, 'r') as z:
    json_files = [f for f in z.namelist()
                  if f.endswith('.json') and '_MACOSX' not in f and not f.endswith('.DS_Store')]

    print(f"Total JSON files found: {len(json_files)}")
    print(f"\nFirst 5 paths:")
    for p in json_files[:5]:
        print(f"  {p}")

    # Inspect the first available JSON
    sample_path = json_files[0]
    print(f"\n{'='*60}")
    print(f"  Inspecting: {sample_path}")
    print(f"{'='*60}")

    with z.open(sample_path) as f:
        data = json.load(f)

    print(f"\n>> Root type: {type(data)}")

    if isinstance(data, list):
        print(f">> Root is a LIST with {len(data)} items")
        if data:
            print(f"\n>> Keys of item[0]:")
            print(list(data[0].keys()) if isinstance(data[0], dict) else type(data[0]))
            print(f"\n>> Full item[0]:")
            print(json.dumps(data[0], indent=2))

    elif isinstance(data, dict):
        print(f">> Root is a DICT")
        print(f"\n>> Top-level keys:")
        print(list(data.keys()))
        print(f"\n>> Full content (first 3000 chars):")
        print(json.dumps(data, indent=2)[:3000])

    # Also inspect a second file to check for schema consistency
    if len(json_files) > 1:
        sample_path_2 = json_files[1]
        print(f"\n{'='*60}")
        print(f"  Cross-checking: {sample_path_2}")
        print(f"{'='*60}")
        with z.open(sample_path_2) as f:
            data2 = json.load(f)
        print(f">> Root type: {type(data2)}")
        if isinstance(data2, dict):
            print(f">> Top-level keys: {list(data2.keys())}")
        elif isinstance(data2, list) and data2:
            print(f">> List length: {len(data2)}")
            print(f">> Keys of item[0]: {list(data2[0].keys()) if isinstance(data2[0], dict) else type(data2[0])}")

Total JSON files found: 10040

First 5 paths:
  ROBUST-MIPS/RobustMIPS/Testing/Stage_1/Proctocolectomy/1/123000/toolposes.json
  ROBUST-MIPS/RobustMIPS/Testing/Stage_1/Proctocolectomy/1/141000/toolposes.json
  ROBUST-MIPS/RobustMIPS/Testing/Stage_1/Proctocolectomy/1/144000/toolposes.json
  ROBUST-MIPS/RobustMIPS/Testing/Stage_1/Proctocolectomy/1/156000/toolposes.json
  ROBUST-MIPS/RobustMIPS/Testing/Stage_1/Proctocolectomy/1/172500/toolposes.json

  Inspecting: ROBUST-MIPS/RobustMIPS/Testing/Stage_1/Proctocolectomy/1/123000/toolposes.json

>> Root type: <class 'list'>
>> Root is a LIST with 2 items

>> Keys of item[0]:
['nodes', 'tags', 'edges', 'transitions']

>> Full item[0]:
{
  "nodes": [
    [
      138.75,
      71.875
    ],
    [
      218.125,
      33.75
    ],
    [
      252.5,
      18.124999999999975
    ],
    null
  ],
  "tags": [
    "visible",
    "visible",
    "visible",
    "missing"
  ],
  "edges": [
    [
      0,
      1
    ],
    [
      1,
      2
    ],
    

In [5]:
#  ROBUST-MIPS — METADATA & SCHEMA DISCOVERY
#  Goal: find any file that maps tool index to instrument name

with zipfile.ZipFile(zip_path, 'r') as z:
    all_files = [f for f in z.namelist() if '_MACOSX' not in f and not f.endswith('.DS_Store')]

    # 1. List all non-image, non-json files (READMEs, CSVs, YAMLs, TXTs, etc.)
    print("=== NON-IMAGE / NON-JSON FILES ===")
    meta_files = [f for f in all_files
                  if not f.endswith(('.png', '.jpg', '.json'))
                  and not f.endswith('/')]
    for f in meta_files:
        print(f"  {f}")

    # 2. Check consistency of list length across 20 random JSON files
    print("\n=== TOOL COUNT PER FRAME (sample of 20) ===")
    import random
    sample = random.sample(json_files, min(20, len(json_files)))
    counts = []
    for jf in sorted(sample)[:20]:
        with z.open(jf) as f:
            d = json.load(f)
        counts.append(len(d))
        print(f"  {jf.split('/')[-3]:>10} | items: {len(d)}")

    from collections import Counter
    print(f"\n>> Item count distribution: {Counter(counts)}")

    # 3. Check node count per tool (to identify skeleton complexity)
    print("\n=== NODE STRUCTURE OF FIRST FILE ===")
    with z.open(json_files[0]) as f:
        data = json.load(f)
    for i, tool in enumerate(data):
        node_count = len(tool['nodes'])
        visible = tool['tags'].count('visible')
        missing = tool['tags'].count('missing')
        print(f"  Tool index {i}: {node_count} nodes | visible={visible} | missing={missing}")
        print(f"    nodes: {tool['nodes']}")

=== NON-IMAGE / NON-JSON FILES ===
  ROBUST-MIPS/RobustMIPS/Testing/Stage_1/Proctocolectomy/1/201000/Thumbs.db
  ROBUST-MIPS/RobustMIPS/Testing/Stage_3/Sigmoid/7/33000/Thumbs.db

=== TOOL COUNT PER FRAME (sample of 20) ===
          10 | items: 2
           7 | items: 1
           7 | items: 2
           9 | items: 2
           9 | items: 0
           9 | items: 1
           9 | items: 1
           9 | items: 0
          10 | items: 2
          10 | items: 2
           8 | items: 1
           9 | items: 1
           9 | items: 2
           1 | items: 1
           8 | items: 3
           8 | items: 1
           8 | items: 0
           9 | items: 1
           9 | items: 0
           9 | items: 0

>> Item count distribution: Counter({1: 8, 2: 6, 0: 5, 3: 1})

=== NODE STRUCTURE OF FIRST FILE ===
  Tool index 0: 4 nodes | visible=3 | missing=1
    nodes: [[138.75, 71.875], [218.125, 33.75], [252.5, 18.124999999999975], None]
  Tool index 1: 4 nodes | visible=4 | missing=0
    nodes: [[310.